# Runbook 5.6.a: Application monitoring and logging  (MODEL SOLUTION, instructor copy, do not distribute)

**Skill:** Detect Attacks · **Maps to:** 5.6.A · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** Implement the SIEM alert rule described in the lesson: given a list of parsed application log
entries, return every IP address whose failed-login (`401`) count exceeds a threshold within a
rolling time window, a classic brute-force signal.

**Lesson objective.** Explain what application logging captures, describe how security teams use log analysis to detect attacks in progress, and identify the key log fields and patterns that signal brute-force, injection, and account-takeover activity.

### What comes in
- `entries`: `list[dict]`, each `{"ip": str, "ts": int, "status": int}` (`ts` = seconds since
  start of log, not necessarily sorted).
- `threshold`: `int`, minimum number of `401` events within the window to flag the IP.
- `window_seconds`: `int`, rolling window size, in seconds (inclusive on both ends).

### What must come out
A **sorted, de-duplicated** `list[str]` of flagged IP addresses.

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```
entries = [
  {"ip":"10.0.0.1","ts":0,"status":401},
  {"ip":"10.0.0.1","ts":10,"status":401},
  {"ip":"10.0.0.1","ts":20,"status":401},
]
solve(entries, threshold=3, window_seconds=60) -> ["10.0.0.1"]
```

```
entries = [{"ip":"10.0.0.2","ts":0,"status":200}]
solve(entries, threshold=1, window_seconds=60) -> []   # success responses never count
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only.
- [ ] Only `status == 401` entries count toward the brute-force count; other codes are ignored.
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(entries: list, threshold: int, window_seconds: int) -> list
```

#### Input
- `entries`: `list[dict]`, each `{"ip": str, "ts": int, "status": int}`. `ts` is seconds since
  the start of the log (non-negative int, not necessarily sorted). `status` is an HTTP status
  code; only `401` entries count toward brute-force detection.
- `threshold`: `int`, minimum number of 401 events within the window to flag the IP.
- `window_seconds`: `int`, inclusive rolling window size, in seconds.

#### Output (`result`)
Sorted, de-duplicated `list[str]` of IP addresses for which some window
`[start, start + window_seconds]` (with `start` equal to one of that IP's 401 timestamps)
contains `>= threshold` 401 events.

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Only entries with `status == 401` count; other status codes are ignored for detection.
- The window bounds are inclusive: an event with `ts == start + window_seconds` counts.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

This is the model solution.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
from collections import defaultdict
from typing import Dict, List


def solve(entries: List[Dict[str, object]], threshold: int, window_seconds: int) -> List[str]:
    by_ip: Dict[str, List[int]] = defaultdict(list)
    for e in entries:
        if e.get("status") == 401:
            by_ip[e["ip"]].append(e["ts"])

    flagged = set()
    for ip, times in by_ip.items():
        times = sorted(times)
        for start in times:
            count = sum(1 for t in times if start <= t <= start + window_seconds)
            if count >= threshold:
                flagged.add(ip)
                break

    return sorted(flagged)


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

def test_brute_force_flagged():
    entries = [
        {"ip": "10.0.0.1", "ts": 0, "status": 401},
        {"ip": "10.0.0.1", "ts": 10, "status": 401},
        {"ip": "10.0.0.1", "ts": 20, "status": 401},
        {"ip": "10.0.0.1", "ts": 30, "status": 401},
        {"ip": "10.0.0.1", "ts": 40, "status": 401},
    ]
    assert impl.solve(entries, threshold=5, window_seconds=60) == ["10.0.0.1"]


def test_successful_logins_not_counted():
    entries = [
        {"ip": "10.0.0.2", "ts": 0, "status": 200},
        {"ip": "10.0.0.2", "ts": 1, "status": 200},
        {"ip": "10.0.0.2", "ts": 2, "status": 200},
    ]
    assert impl.solve(entries, threshold=2, window_seconds=60) == []


def test_below_threshold_not_flagged():
    entries = [
        {"ip": "10.0.0.3", "ts": 0, "status": 401},
        {"ip": "10.0.0.3", "ts": 5, "status": 401},
    ]
    assert impl.solve(entries, threshold=5, window_seconds=60) == []


def test_multiple_ips_flagged_sorted():
    entries = [
        {"ip": "10.0.0.9", "ts": 0, "status": 401},
        {"ip": "10.0.0.9", "ts": 1, "status": 401},
        {"ip": "10.0.0.5", "ts": 0, "status": 401},
        {"ip": "10.0.0.5", "ts": 1, "status": 401},
    ]
    assert impl.solve(entries, threshold=2, window_seconds=60) == ["10.0.0.5", "10.0.0.9"]


def test_events_too_far_apart_not_flagged():
    entries = [
        {"ip": "10.0.0.4", "ts": 0, "status": 401},
        {"ip": "10.0.0.4", "ts": 120, "status": 401},
    ]
    assert impl.solve(entries, threshold=2, window_seconds=60) == []


def test_window_boundary_inclusive():
    entries = [
        {"ip": "10.0.0.6", "ts": 0, "status": 401},
        {"ip": "10.0.0.6", "ts": 60, "status": 401},
    ]
    assert impl.solve(entries, threshold=2, window_seconds=60) == ["10.0.0.6"]


def test_unsorted_input():
    entries = [
        {"ip": "10.0.0.7", "ts": 40, "status": 401},
        {"ip": "10.0.0.7", "ts": 0, "status": 401},
        {"ip": "10.0.0.7", "ts": 20, "status": 401},
    ]
    assert impl.solve(entries, threshold=3, window_seconds=60) == ["10.0.0.7"]


def test_mixed_status_only_401_counts():
    entries = [
        {"ip": "10.0.0.8", "ts": 0, "status": 401},
        {"ip": "10.0.0.8", "ts": 1, "status": 200},
        {"ip": "10.0.0.8", "ts": 2, "status": 403},
        {"ip": "10.0.0.8", "ts": 3, "status": 401},
    ]
    assert impl.solve(entries, threshold=2, window_seconds=60) == ["10.0.0.8"]


def test_empty_entries():
    assert impl.solve([], threshold=1, window_seconds=60) == []


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Run your finished tool on the examples from the brief, then change one value and predict the answer before you run it again.

- [ ] Every example says OK
- [ ] Your own changed example gives the answer you predicted

In [ ]:
checks = [
    (lambda: solve([], threshold=1, window_seconds=60),
     []),
]

for n, (run, want) in enumerate(checks, 1):
    try:
        got = run()
    except Exception as e:
        got = "error: " + repr(e)
    print("example", n, "OK  " if got == want else "DIFF", "got:", got, " expected:", want)


## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it